# Evidencia de aprendizaje 3 – Optimización de procesos de desarrollo
**Aplicación web del iPhone 18 Pro y 18 Pro Max: CI/CD con GitHub Actions y despliegue en GitHub Pages**

- **Estudiante:** Londoño Acebedo Camilo
- **Curso:** PREICA2602B020048 – Programación para Análisis de Datos (IU Digital)
- **Repositorio:** https://github.com/camilolondono-IUD/scraping-iphone-ishop
- **Aplicación publicada:** https://camilolondono-iud.github.io/scraping-iphone-ishop/

Este notebook continúa las Evidencias 1 y 2. El scraper (EA1) se llevó a un repositorio con un
pipeline CI/CD (EA2) y en esta evidencia se convirtió en una **aplicación web** configurada por
variables de entorno y desplegada automáticamente en la nube. Aquí se **consume la aplicación
desplegada**: se verifica la versión publicada, se descargan sus datos, se analizan y se revisan
las ejecuciones del pipeline en GitHub Actions.

## 1. Librerías y configuración

In [ ]:
import json
import os
import time

import requests
import pandas as pd
import matplotlib.pyplot as plt

# URL de la aplicación desplegada por el pipeline (se puede sobrescribir con una variable de entorno)
URL_APP = os.environ.get('URL_APP', 'https://camilolondono-iud.github.io/scraping-iphone-ishop/').rstrip('/') + '/'
REPO = 'camilolondono-IUD/scraping-iphone-ishop'
VARIANTES_ESPERADAS = 32

sesion = requests.Session()
sesion.headers.update({'User-Agent': 'notebook-EA3'})

def obtener(ruta):
    """Descarga un recurso de la app desplegada, evitando la caché de la CDN."""
    r = sesion.get(URL_APP + ruta, params={'nocache': time.time_ns()}, timeout=30)
    r.raise_for_status()
    return r

## 2. Verificar el despliegue
`version.json` lo genera el job **build** del pipeline con las variables de entorno `APP_ENV` y
`APP_VERSION` (definidas en *Settings > Secrets and variables > Actions > Variables*) y el commit
(`GITHUB_SHA`). Es la misma comprobación que hace el job **verificar** después de cada despliegue.

In [ ]:
inicio = time.time()
r_index = obtener('index.html')
version = obtener('version.json').json()
latencia = time.time() - inicio

print('index.html ->', r_index.status_code, f'({len(r_index.content):,} bytes)')
print('Tiempo de respuesta:', round(latencia, 2), 's')
pd.Series(version, name='Valor').to_frame()

## 3. Consumir los datos publicados por la aplicación (`datos.json`)

In [ ]:
df = pd.DataFrame(obtener('datos.json').json())
print(f'Variantes publicadas: {len(df)} (esperadas: {VARIANTES_ESPERADAS})')
assert len(df) == VARIANTES_ESPERADAS, 'La app no publica todas las variantes'
df.head()

## 4. Análisis de los datos

In [ ]:
def orden_capacidad(cap):
    n, u = str(cap).split()
    return int(n) * (1024 if u == 'TB' else 1)

caps = sorted(df['Capacidad'].unique(), key=orden_capacidad)
tabla = df.pivot_table(index='Capacidad', columns='Modelo', values='Precio_COP', aggfunc='mean').reindex(caps)
tabla['Diferencia Pro Max - Pro'] = tabla['iPhone 18 Pro Max'] - tabla['iPhone 18 Pro']
tabla.map(lambda v: f'${v:,.0f}')

In [ ]:
disp = pd.crosstab([df['Modelo'], df['Capacidad']], df['Disponibilidad']).reindex(caps, level=1)
resumen = {
    'Precio mínimo': f"${df['Precio_COP'].min():,.0f}",
    'Precio máximo': f"${df['Precio_COP'].max():,.0f}",
    'Disponibles': int((df['Disponibilidad'] == 'Disponible').sum()),
    'Agotadas': int((df['Disponibilidad'] == 'Agotado').sum()),
    '¿El color cambia el precio?': 'No' if (df.groupby(['Modelo', 'Capacidad'])['Precio_COP'].nunique() == 1).all() else 'Sí',
}
display(pd.Series(resumen, name='Valor').to_frame())
disp

In [ ]:
ax = (tabla[['iPhone 18 Pro', 'iPhone 18 Pro Max']] / 1e6).plot(kind='bar', rot=0, figsize=(8, 4.5),
                                                               color=['#9ab8d8', '#2f6ea8'])
for cont in ax.containers:
    ax.bar_label(cont, labels=[f'{v:.2f}'.replace('.', ',') for v in cont.datavalues], fontsize=8)
ax.set_title('Precio por capacidad – datos de la aplicación desplegada')
ax.set_xlabel('Capacidad'); ax.set_ylabel('Precio (millones de COP)')
plt.tight_layout(); plt.show()

## 5. Ejecuciones del pipeline en GitHub Actions
La API pública de GitHub permite consultar las últimas ejecuciones del workflow **CI/CD** y sus
jobs (`test`, `build`, `deploy`, `verificar`).

In [ ]:
api = f'https://api.github.com/repos/{REPO}/actions/runs'
runs = sesion.get(api, params={'per_page': 8}, timeout=30).json().get('workflow_runs', [])
ejecuciones = pd.DataFrame([{
    'Ejecución': r['run_number'], 'Evento': r['event'], 'Rama': r['head_branch'],
    'Commit': r['head_sha'][:7], 'Estado': r['conclusion'] or r['status'],
    'Fecha (UTC)': r['created_at'].replace('T', ' ').rstrip('Z'),
} for r in runs])
ejecuciones

In [ ]:
# Jobs de la última ejecución en main
ultima = next(r for r in runs if r['head_branch'] == 'main' and r['event'] == 'push')
jobs = sesion.get(ultima['jobs_url'], timeout=30).json()['jobs']
pd.DataFrame([{
    'Job': j['name'], 'Resultado': j['conclusion'],
    'Duración (s)': (pd.Timestamp(j['completed_at']) - pd.Timestamp(j['started_at'])).total_seconds(),
} for j in jobs])

## 6. Conclusiones
- La aplicación desplegada responde y publica la versión y el commit que generó el pipeline, lo que
  permite rastrear qué código está en producción.
- Los datos se consumen desde la app (`datos.json`) igual que lo haría cualquier otro cliente: el
  despliegue continuo convierte el resultado del scraping en un servicio accesible.
- El pipeline automatiza pruebas, construcción, despliegue y verificación; cada merge a `main`
  actualiza la aplicación sin pasos manuales.